In [ ]:
# 2D CNN — EfficientNet-B0 on HMS spectrograms (WORKSHOP VERSION, self-contained)
# Single-stage training, workshop-scale data (600 train / 120 val rows)
#
# Rationale for no 2-step: 2-step training rescues minority-class knowledge from a large,
# class-imbalanced dataset (Seizure ~256 vs Other ~3050 in the full data). The workshop subset
# is already balanced (100 rows/class train, 20 rows/class val), so there is no imbalance to
# correct for — a single training stage is sufficient here.
#
# This notebook is fully self-contained — no external .py imports. All classes and functions
# are defined inline below so you can read, tweak, and re-run any part of the pipeline directly.
!pip install timm -q

In [ ]:
import os, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
import timm

IS_KAGGLE = os.path.exists('/kaggle')
print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'} | torch {torch.__version__}")

In [ ]:
# ============ Config ============
from dataclasses import dataclass, field

@dataclass
class Config2D:
    # ── Data paths ──────────────────────────────────────────────
    spectrogram_dir: str = "train_spectrograms"
    spec_cache_dir:  str = "/kaggle/working/spec_cache"

    # ── Spectrogram image dimensions ────────────────────────────────────────
    img_height: int = 100   # freq bins per chain (4 chains stacked → 400 total rows)
    img_width:  int = 300   # time columns per window (2-s resolution → 300 = 600 s)

    # ── Model ───────────────────────────────────────────────────────
    backbone:    str  = "efficientnet_b0"
    pretrained:  bool = True
    num_classes: int  = 6

    # ── Training ──────────────────────────────────────────────
    batch_size:   int   = 32
    lr:           float = 1e-3
    weight_decay: float = 1e-4
    drop_rate:    float = 0.3

    # ── Early stopping ───────────────────────────────────────────
    patience: int = 10

    # ── Misc ──────────────────────────────────────────────────────
    seed:        int = 42
    num_workers: int = 2

    device: str = field(init=False)

    def __post_init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"


cfg = Config2D()

if IS_KAGGLE:
    DATA_ROOT       = '/kaggle/input/competitions/hms-harmful-brain-activity-classification'
    RAW_TRAIN_PATH  = os.path.join(DATA_ROOT, 'train.csv')
    SAMPLE_IDS_PATH = '/kaggle/input/datasets/xiaosufrankhu/midas-summer-academy-wk3-eeg/workshop_sample_ids.csv'
    cfg.spectrogram_dir = os.path.join(DATA_ROOT, 'train_spectrograms')
else:
    DATA_ROOT       = os.path.abspath('../')
    RAW_TRAIN_PATH  = os.path.abspath('../data_raw/train.csv')
    SAMPLE_IDS_PATH = os.path.abspath('../data_raw/workshop_sample_ids.csv')
    cfg.spectrogram_dir = os.path.join(DATA_ROOT, 'train_spectrograms')

# ── reproducibility ───────────────────────────────────────────────────────────
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
torch.cuda.manual_seed_all(cfg.seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE  = torch.device(cfg.device)
USE_AMP = DEVICE.type == 'cuda'

print(cfg)
print(f"Device: {DEVICE} | AMP: {USE_AMP}")

# NOTE: if running on Kaggle and this path doesn't exist, run !ls /kaggle/input
# and update DATA_ROOT to match how the competition data was attached.
print(f"Competition data path exists: {os.path.exists(DATA_ROOT)}")

In [ ]:
# ============ Data loading (workshop subset) ============
# The raw HMS train.csv carries everything SpectrogramDataset needs (spectrogram_id,
# spectrogram_label_offset_seconds, vote columns). workshop_sample_ids.csv only carries
# ID + split, so we merge the two to reconstruct full rows for the 600/120 workshop subset.

raw_df     = pd.read_csv(RAW_TRAIN_PATH)
sample_ids = pd.read_csv(SAMPLE_IDS_PATH)

merged = raw_df.merge(
    sample_ids[["eeg_id", "eeg_sub_id", "split"]],
    on=["eeg_id", "eeg_sub_id"],
    how="inner",
)
train_df = merged[merged["split"] == "train"].reset_index(drop=True)
val_df   = merged[merged["split"] == "val"].reset_index(drop=True)

print(f'Train: {len(train_df):,} rows')
print(f'Val   : {len(val_df):,} rows')
print(train_df['expert_consensus'].value_counts())

In [ ]:
# ============ Spectrogram preprocessing ============
# Kaggle's precomputed spectrograms already have 4 bipolar chains (LL, RL, LP, RP),
# each with 100 frequency bins, stacked into 400 rows per parquet file.
# We convert each {spec_id}.parquet → {spec_id}.npy once, then cache it.

def preprocess_one_spectrogram(spec_id: int, spectrogram_dir: str, cache_dir: str) -> None:
    dst = os.path.join(cache_dir, f"{spec_id}.npy")
    if os.path.exists(dst):
        return
    src = os.path.join(spectrogram_dir, f"{spec_id}.parquet")
    df  = pd.read_parquet(src)
    df  = df.fillna(0)
    if "time" in df.columns:
        df = df.drop(columns=["time"])
    # (total_time, 400) → transpose → (400, total_time)
    arr = df.to_numpy(dtype=np.float32).T
    np.save(dst, arr)


os.makedirs(cfg.spec_cache_dir, exist_ok=True)

# Workshop mode only needs the spectrograms actually used by the 720-row subset —
# far fewer than the full ~11,000 files, which keeps this step fast (seconds, not minutes).
needed_ids = set(train_df['spectrogram_id']).union(val_df['spectrogram_id'])
for spec_id in needed_ids:
    preprocess_one_spectrogram(int(spec_id), cfg.spectrogram_dir, cfg.spec_cache_dir)

print(f'Cache ready: {len(needed_ids)} spectrograms for workshop subset')

In [ ]:
# ============ Dataset ============
VOTE_COLS = ["seizure_vote", "lpd_vote", "gpd_vote",
             "lrda_vote",    "grda_vote", "other_vote"]


class SpectrogramDataset(Dataset):
    """
    Loads pre-cached (400, total_time) .npy spectrograms, extracts a 300-column
    (600-second) window around the labeled offset, normalizes it, and reshapes
    the 400 stacked rows back into 4 separate bipolar chains: (4, 100, 300).
    """

    def __init__(self, metadata_df: pd.DataFrame, spec_cache_dir: str):
        self.meta           = metadata_df.reset_index(drop=True)
        self.spec_cache_dir = spec_cache_dir

        votes    = self.meta[VOTE_COLS].to_numpy(dtype=np.float32)
        row_sums = votes.sum(axis=1, keepdims=True)
        row_sums = np.where(row_sums == 0, 1.0, row_sums)
        self.soft_labels = votes / row_sums          # (N, 6)
        self.hard_labels = self.soft_labels.argmax(1)  # (N,)

    def __len__(self) -> int:
        return len(self.meta)

    def __getitem__(self, idx: int) -> dict:
        row     = self.meta.iloc[idx]
        spec_id = int(row["spectrogram_id"])
        offset  = float(row["spectrogram_label_offset_seconds"])

        spec = np.load(os.path.join(self.spec_cache_dir, f"{spec_id}.npy"))

        # extract 300-column window; 2-second time resolution → col = offset // 2
        col_start = int(offset // 2)
        window    = spec[:, col_start:col_start + 300]   # (400, ≤300)

        # right-pad to exactly 300 columns if the window runs off the end
        if window.shape[1] < 300:
            pad    = 300 - window.shape[1]
            window = np.pad(window, ((0, 0), (0, pad)), mode="constant")

        # normalize: clip → log → z-score
        window = np.clip(window, np.exp(-4), np.exp(8))
        window = np.log(window)
        mu     = window.mean()
        sigma  = window.std()
        window = (window - mu) / (sigma + 1e-6)

        # split 400 stacked rows back into 4 chains: (400, 300) → (4, 100, 300)
        window = window.reshape(4, 100, 300)

        img   = torch.from_numpy(window).float()
        soft  = torch.from_numpy(self.soft_labels[idx])
        label = int(self.hard_labels[idx])

        return {"image": img, "soft_label": soft, "label": label}

In [ ]:
# ============ Model ============
class EfficientNetEEG(nn.Module):
    """
    timm EfficientNet-B0 backbone adapted for 4-channel spectrogram input
    (instead of the usual 3-channel RGB), with a linear head for 6 classes.
    Pretrained on ImageNet — note the domain gap: natural images vs EEG spectrograms.
    """

    def __init__(self, backbone="efficientnet_b0", num_classes=6,
                 pretrained=True, drop_rate=0.3):
        super().__init__()
        self.encoder = timm.create_model(
            backbone,
            pretrained=pretrained,
            in_chans=4,       # 4 bipolar chains instead of RGB
            num_classes=0,    # remove timm's default head; add our own
            drop_rate=drop_rate,
        )
        n_features = self.encoder.num_features
        self.head = nn.Linear(n_features, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.encoder(x)
        return self.head(features)


model = EfficientNetEEG(
    backbone    = cfg.backbone,
    num_classes = cfg.num_classes,
    pretrained  = cfg.pretrained,
    drop_rate   = cfg.drop_rate,
).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Backbone    : {cfg.backbone}")
print(f"Parameters  — total: {total_params:,} | trainable: {trainable_params:,}")

In [ ]:
# ============ Single-stage training ============
GRAD_CLIP  = 1.0
NUM_EPOCHS = 5   # ~5-10 min on Kaggle T4 for this subset size
LR         = cfg.lr

criterion     = nn.KLDivLoss(reduction='batchmean')
val_criterion = nn.CrossEntropyLoss()
scaler        = torch.amp.GradScaler('cuda', enabled=USE_AMP)

history = []

train_ds     = SpectrogramDataset(train_df, cfg.spec_cache_dir)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, pin_memory=True)
val_ds       = SpectrogramDataset(val_df, cfg.spec_cache_dir)
val_loader   = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=True)
print(f'Train: {len(train_ds):,} | Val: {len(val_ds):,} | Epochs: {NUM_EPOCHS}')


def validate_2d(model, loader):
    model.eval()
    val_ce, val_kl = 0.0, 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            x    = batch['image'].to(DEVICE)
            y    = batch['label'].to(DEVICE)
            soft = batch['soft_label'].to(DEVICE)
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                logits = model(x)
            val_ce += val_criterion(logits, y).item()
            val_kl += criterion(F.log_softmax(logits, dim=1), soft).item()
            all_preds .extend(logits.argmax(1).cpu().tolist())
            all_labels.extend(y.cpu().tolist())
    macro_f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    per_class = f1_score(all_labels, all_preds, average=None,    zero_division=0)
    return (val_kl / len(loader), val_ce / len(loader), macro_f1, per_class)


optimizer = AdamW(model.parameters(), lr=LR, weight_decay=cfg.weight_decay)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
# NOTE: "best" is defined by val KL (the competition metric), matching the XGBoost
# and 1D CNN notebooks, so all three models are selected the same way.
best_kl, best_f1, best_epoch, wait = float('inf'), 0.0, 0, 0
best_state = None

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    t0 = time.time()
    train_loss = 0.0
    for batch in train_loader:
        x    = batch['image'].to(DEVICE)
        soft = batch['soft_label'].to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            logits = model(x)
            loss   = criterion(F.log_softmax(logits, dim=1), soft)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
    scheduler.step()

    val_kl, val_ce, macro_f1, _ = validate_2d(model, val_loader)
    avg_train = train_loss / len(train_loader)
    elapsed   = time.time() - t0

    history.append({
        'epoch': epoch, 'train_kl': avg_train,
        'val_kl': val_kl, 'val_ce': val_ce, 'macro_f1': macro_f1,
    })
    print(f'Epoch {epoch:03d} | train_kl {avg_train:.4f} | '
          f'val_kl {val_kl:.4f} | val_ce {val_ce:.4f} | '
          f'macro_f1 {macro_f1:.4f} | {elapsed:.0f}s')

    if val_kl < best_kl:
        best_kl, best_f1, best_epoch, wait = val_kl, macro_f1, epoch, 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        torch.save(model.state_dict(), 'best_2d_workshop.pt')
        print(f'  \u2713 saved best_2d_workshop.pt (val_kl={best_kl:.4f})')
    else:
        wait += 1
        if wait >= cfg.patience:
            print(f'Early stopping at epoch {epoch}')
            break

print(f'\nTraining complete \u2014 best val_kl={best_kl:.4f} (macro_f1={best_f1:.4f}) at epoch {best_epoch}')
if best_state is not None:
    model.load_state_dict(best_state)

_, _, _, best_per_class = validate_2d(model, val_loader)  # per-class F1 at the KL-best checkpoint


In [ ]:
# ============ Final report (standard format) ============
CLASS_NAMES = ["Seizure", "LPD", "GPD", "LRDA", "GRDA", "Other"]

epochs_   = [h['epoch']    for h in history]
train_kl_ = [h['train_kl'] for h in history]
val_kl_   = [h['val_kl']   for h in history]
val_ce_   = [h['val_ce']   for h in history]
macro_f1_ = [h['macro_f1'] for h in history]

print(f"Val macro F1     : {best_f1:.4f}")
print(f"Val KL divergence: {best_kl:.4f}")
print(f"(random-guess baseline for 6 balanced classes: macro F1 \u2248 0.167)")
print("\nPer-class F1 (best epoch):")
for name, f in zip(CLASS_NAMES, best_per_class):
    print(f"  {name:<10} {f:.4f}")

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

axes[0, 0].plot(epochs_, train_kl_, label='train KL (loss)')
axes[0, 0].plot(epochs_, val_kl_,   label='val KL (loss)')
axes[0, 0].axvline(best_epoch, color='red', linestyle='--', label=f'best={best_epoch}')
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('KL Divergence (loss)')
axes[0, 0].set_title('Train/Val KL loss'); axes[0, 0].legend(fontsize=8)

axes[0, 1].plot(epochs_, val_ce_, color='orange')
axes[0, 1].axvline(best_epoch, color='red', linestyle='--', label=f'best={best_epoch}')
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Cross-Entropy Loss')
axes[0, 1].set_title('Val CE (monitoring)'); axes[0, 1].legend(fontsize=8)

axes[1, 0].plot(epochs_, macro_f1_, color='seagreen')
axes[1, 0].axhline(1/6, color='gray', linestyle='--', linewidth=1, label='random guess')
axes[1, 0].axvline(best_epoch, color='red', linestyle='--', label=f'best={best_epoch}')
axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('Val Macro F1')
axes[1, 0].set_title('Val Macro F1'); axes[1, 0].legend(fontsize=8)

axes[1, 1].bar(CLASS_NAMES, best_per_class, color='steelblue')
axes[1, 1].axhline(best_f1, color='red', linestyle='--', label=f'macro F1 = {best_f1:.3f}')
axes[1, 1].set_ylim(0, 1); axes[1, 1].set_ylabel('F1')
axes[1, 1].set_title(f'Per-class F1 (val, epoch {best_epoch})'); axes[1, 1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('training_curves_2d_workshop.png', dpi=150)
plt.show()


## Tweak & Compare (~20 min)

Pick 1-2 changes below, rerun the cell, and log your result in the shared sheet.

| Knob | Default | Try |
|---|---|---|
| learning rate | 1e-3 | 3e-4 / 3e-3 |
| `drop_rate` | 0.3 | 0.1 / 0.5 |
| time window (`img_width`) | 300 (600 s) | 150 (300 s) / 450 (900 s) |

**Log your run:** What you changed | Val F1 | Val KL | one-line observation


In [ ]:
# ============ Tweak & Compare ============
TWEAK_LR         = 1e-3   # <- change me (try 3e-4 or 3e-3)
TWEAK_DROP_RATE  = 0.3    # <- change me (try 0.1 or 0.5)
TWEAK_IMG_WIDTH  = 300    # <- change me (try 150 or 450)

# re-seed locally so this cell gives the same result every time it's rerun on its own,
# regardless of how many earlier cells (and how much of the global RNG stream) ran first
SEED = Config2D.seed
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)


class SpectrogramDatasetTweak(SpectrogramDataset):
    """Same as SpectrogramDataset, but with a configurable time-window width
    (instead of the fixed 300 columns / 600 seconds)."""

    def __init__(self, metadata_df, spec_cache_dir, width=300):
        self.width = width
        super().__init__(metadata_df, spec_cache_dir)

    def __getitem__(self, idx):
        row     = self.meta.iloc[idx]
        spec_id = int(row["spectrogram_id"])
        offset  = float(row["spectrogram_label_offset_seconds"])

        spec = np.load(os.path.join(self.spec_cache_dir, f"{spec_id}.npy"))
        col_start = int(offset // 2)
        window    = spec[:, col_start:col_start + self.width]

        if window.shape[1] < self.width:
            pad = self.width - window.shape[1]
            window = np.pad(window, ((0, 0), (0, pad)), mode="constant")

        # normalize: clip -> log -> z-score (same as SpectrogramDataset above —
        # missing this step is what caused the earlier Val KL: inf / F1: 0.0000 bug:
        # raw un-logged spectrogram power values are unbounded and blow up the
        # pretrained EfficientNet's activations within a few batches)
        window = np.clip(window, np.exp(-4), np.exp(8))
        window = np.log(window)
        mu     = window.mean()
        sigma  = window.std()
        window = (window - mu) / (sigma + 1e-6)

        image = window.reshape(4, cfg.img_height, self.width).astype(np.float32)
        image = np.nan_to_num(image, nan=0.0, posinf=0.0, neginf=0.0)

        return {
            "image":      torch.from_numpy(image),
            "label":      int(self.hard_labels[idx]),
            "soft_label": torch.from_numpy(self.soft_labels[idx]),
        }


tweak_train_ds = SpectrogramDatasetTweak(train_df, cfg.spec_cache_dir, width=TWEAK_IMG_WIDTH)
tweak_val_ds   = SpectrogramDatasetTweak(val_df,   cfg.spec_cache_dir, width=TWEAK_IMG_WIDTH)
tweak_train_loader = DataLoader(tweak_train_ds, batch_size=cfg.batch_size, shuffle=True,
                                 num_workers=cfg.num_workers, pin_memory=True)
tweak_val_loader   = DataLoader(tweak_val_ds,   batch_size=cfg.batch_size, shuffle=False,
                                 num_workers=cfg.num_workers, pin_memory=True)

tweak_model = EfficientNetEEG(
    backbone=cfg.backbone, num_classes=cfg.num_classes,
    pretrained=cfg.pretrained, drop_rate=TWEAK_DROP_RATE,
).to(DEVICE)
tweak_optimizer = AdamW(tweak_model.parameters(), lr=TWEAK_LR, weight_decay=cfg.weight_decay)
tweak_scheduler = CosineAnnealingLR(tweak_optimizer, T_max=NUM_EPOCHS)
tweak_scaler    = torch.amp.GradScaler('cuda', enabled=USE_AMP)

t_best_kl, t_best_f1, t_wait = float('inf'), 0.0, 0
for epoch in range(1, NUM_EPOCHS + 1):
    tweak_model.train()
    for batch in tweak_train_loader:
        x    = batch['image'].to(DEVICE)
        soft = batch['soft_label'].to(DEVICE)
        tweak_optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            logits = tweak_model(x)
            loss   = criterion(F.log_softmax(logits, dim=1), soft)
        tweak_scaler.scale(loss).backward()
        tweak_scaler.unscale_(tweak_optimizer)
        nn.utils.clip_grad_norm_(tweak_model.parameters(), GRAD_CLIP)
        tweak_scaler.step(tweak_optimizer)
        tweak_scaler.update()
    tweak_scheduler.step()

    t_val_kl, _, t_macro_f1, _ = validate_2d(tweak_model, tweak_val_loader)
    if t_val_kl < t_best_kl:
        t_best_kl, t_best_f1, t_wait = t_val_kl, t_macro_f1, 0
    else:
        t_wait += 1
        if t_wait >= cfg.patience:
            break

print(f"lr={TWEAK_LR} | drop_rate={TWEAK_DROP_RATE} | img_width={TWEAK_IMG_WIDTH}")
print(f"Val macro F1 : {t_best_f1:.4f}")
print(f"Val KL       : {t_best_kl:.4f}")
print("-> log this row in the shared sheet: what you changed / F1 / KL / one-line observation")


### Done early? Extend (~10 min)

**What this does:** ensembles this 2D CNN's predictions with the 1D CNN's (domain-informed)
predictions, and checks whether the combination beats either model alone. This is one of the
most consistent findings across the top-10 Kaggle solutions review earlier — and here you get
to see it happen with two models built from genuinely different information sources
(time-domain waveform vs. frequency-domain spectrogram of the *same* labeled event).

This step involves a bit of cross-notebook plumbing (matching two independently-loaded
validation sets by ID, reusing ground-truth labels from this notebook's own val set since the
1D CNN's exported file doesn't carry them) that isn't a great fit for a quick AI-prompt exercise
at this point in a long day — so unlike the earlier Tweak & Extend steps, just **run the cell
below directly** and read through what it's doing.

A pre-computed `cnn1d_domain_informed_val_probs.csv` (from the 1D CNN notebook's export cell)
needs to be attached to this notebook as a dataset for this to work. Update the path below to
match — Kaggle attaches datasets under `/kaggle/input/<owner>/<dataset-slug>/`; check the file
browser panel on the right if the path doesn't match.


In [ ]:
# ============ Ensemble: 2D CNN + 1D CNN ============
CNN1D_PROBS_PATH = "/kaggle/input/datasets/xiaosufrankhu/midas-summer-academy-wk3-eeg/cnn1d_domain_informed_val_probs.csv"
cnn1d_probs_df = pd.read_csv(CNN1D_PROBS_PATH)

# ---- get this 2D CNN's own val-set probabilities (using the KL-best checkpoint,
#      already loaded back into `model` at the end of the Single-stage training cell) ----
model.eval()
cnn2d_ids, cnn2d_probs, cnn2d_soft = [], [], []
with torch.no_grad():
    for batch in val_loader:
        x = batch['image'].to(DEVICE)
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            logits = model(x)
        cnn2d_probs.append(F.softmax(logits, dim=1).cpu().numpy())
        cnn2d_soft.append(batch['soft_label'].numpy())
cnn2d_probs = np.concatenate(cnn2d_probs, axis=0)   # (120, 6)
cnn2d_soft  = np.concatenate(cnn2d_soft,  axis=0)   # (120, 6), true vote distribution

prob_cols = [f"prob_{c.lower()}" for c in CLASS_NAMES]
cnn2d_ids_df = val_df[["eeg_id", "eeg_sub_id"]].reset_index(drop=True)
cnn2d_df = pd.concat([cnn2d_ids_df, pd.DataFrame(cnn2d_probs, columns=prob_cols)], axis=1)

# ---- merge on ID rather than assuming row order matches — the two notebooks
#      load/order the validation set independently ----
merged = cnn2d_df.merge(cnn1d_probs_df, on=["eeg_id", "eeg_sub_id"], suffixes=("_2d", "_1d"))
assert len(merged) == len(cnn2d_df), "some validation rows didn't find a 1D CNN match — check the CSV path/upload"

probs_2d = merged[[f"{c}_2d" for c in prob_cols]].to_numpy()
probs_1d = merged[[f"{c}_1d" for c in prob_cols]].to_numpy()
probs_ensemble = (probs_2d + probs_1d) / 2

# re-derive the true soft-label distribution in the merged row order
soft_lookup = dict(zip(zip(cnn2d_ids_df["eeg_id"], cnn2d_ids_df["eeg_sub_id"]),
                        list(cnn2d_soft)))
soft_merged = np.stack([soft_lookup[(r.eeg_id, r.eeg_sub_id)] for r in merged.itertuples()])
hard_merged = soft_merged.argmax(axis=1)


def _f1_kl(probs, hard_labels, soft_labels):
    f1 = f1_score(hard_labels, probs.argmax(axis=1), average="macro", zero_division=0)
    kl = (soft_labels * np.log(np.clip(soft_labels, 1e-7, 1) / np.clip(probs, 1e-7, 1))).sum(axis=1).mean()
    return f1, kl


f1_2d, kl_2d = _f1_kl(probs_2d, hard_merged, soft_merged)
f1_1d, kl_1d = _f1_kl(probs_1d, hard_merged, soft_merged)
f1_ens, kl_ens = _f1_kl(probs_ensemble, hard_merged, soft_merged)

print(f"{'Model':<20}{'Val F1':>10}{'Val KL':>10}")
print(f"{'-'*40}")
print(f"{'2D CNN alone':<20}{f1_2d:>10.4f}{kl_2d:>10.4f}")
print(f"{'1D CNN alone':<20}{f1_1d:>10.4f}{kl_1d:>10.4f}")
print(f"{'Ensemble (avg)':<20}{f1_ens:>10.4f}{kl_ens:>10.4f}")
